## DeepRetinotopy toolbox

This interactive notebook demonstrates how to predict retinotopic maps from anatomical data using a geometric deep learning model of retinotopy. For more details about the toolbox, please check our [repository](https://github.com/felenitaribeiro/deepRetinotopy_Toolbox) and [webpage](https://felenitaribeiro.github.io/deepRetinotopy_TheToolbox/). 
Requirements:

1. FreeSurfer directory;
2. HCP "fs_LR-deformed_to-fsaverage" surfaces, available [here](https://github.com/Washington-University/HCPpipelines/tree/master/global/templates/standard_mesh_atlases/resample_fsaverage).


### Citation and Resources
#### Tools included in this workflow
__deepretinotopy__:
- Ribeiro, F. L. et al. (2025). Predicting functional topography of the human visual cortex from cortical anatomy at scale. [bioRxiv](https://www.biorxiv.org/content/10.1101/2025.11.27.690210v2)


## Load Software Tools

In [ ]:
%%capture 
!pip install nilearn nibabel numpy

In [ ]:
# we can use module to load deepretinotopy in a specific version
import module
import os
await module.load('deepretinotopy/1.0.18')
await module.list()

In [ ]:
# Request a freesurfer license and store it in your homedirectory. This is just an example - please replace with your license id:
!echo "fernanda.ribeiro@uni-giessen.de" > ~/.license
!echo "12345" >> ~/.license
!echo "*CxxxXXxxXXxx" >> ~/.license
!echo "FS/XXXXXXXXXX" >> ~/.license

## Loading your data

If you are running this notebook on NeurodeskPlay, the easiest way to transfer your data to the Cloud instance is to drag and drop it. DeepRetinotopy only requires a minimal subset of FreeSurfer outputs:

### Essential Files
```
<subject_id>/surf/
├── lh.white          # White matter surface
├── lh.pial           # Pial surface  
├── lh.sphere.reg     # Spherical surface for resampling
├── rh.white          # White matter surface
├── rh.pial           # Pial surface
└── rh.sphere.reg     # Spherical surface for resampling
```

So, you can simply drag and drop a .zip file with those files, maintaining the FreeSurfer directory structure. 

**_Note_**: The toolbox automatically generates the midthickness surfaces and curvature data from these base files during processing. All other FreeSurfer outputs (thickness maps, parcellations, etc.) are not required for retinotopic mapping prediction.

Since we have preprocessed data available, the only remaining requirements are the HCP template surfaces. 
Thus, we need to create a "templates" folder (for example, ./templates/) and download the following files from the link above:

- "fs_LR-deformed_to-fsaverage.R.sphere.32k_fs_LR.surf.gii";
- "fs_LR-deformed_to-fsaverage.L.sphere.32k_fs_LR.surf.gii";

In [ ]:
# Set up paths relative to the current directory
os.environ['PATH_FREESURFER_DIR'] = "ADD HERE THE PATH TO YOUR FREESURFER DIR"
os.environ['PATH_HCP_TEMPLATE_SURFACE'] = "/home/jovyan/Desktop/deepRetinotopy_workshop/notebooks/data/templates"
os.environ['DATASET_NAME'] = "my_dataset"
os.environ['MAPS'] = "polarAngle,eccentricity"

In [ ]:
# Create templates directory
os.makedirs(os.environ['PATH_HCP_TEMPLATE_SURFACE'], exist_ok=True)

In [ ]:
# Download templates to the relative path
!cd $PATH_HCP_TEMPLATE_SURFACE && \
wget -nc https://github.com/Washington-University/HCPpipelines/raw/master/global/templates/standard_mesh_atlases/resample_fsaverage/fs_LR-deformed_to-fsaverage.L.sphere.32k_fs_LR.surf.gii && \
wget -nc https://github.com/Washington-University/HCPpipelines/raw/master/global/templates/standard_mesh_atlases/resample_fsaverage/fs_LR-deformed_to-fsaverage.R.sphere.32k_fs_LR.surf.gii

In [ ]:
os.environ['SUBJECT_ID'] = "ADD A SUBJECT ID HERE"

## Running deepRetinotopy for a single subject

Here, we illustrate how to run deepRetinotopy for a single subject. Note that, if no output directory is provided, all maps are saved within each subject subdirectory in the freesurfer directory. 

In [ ]:
## Process single subject without an explicit output directory
!deepRetinotopy -s $PATH_FREESURFER_DIR -t $PATH_HCP_TEMPLATE_SURFACE -d $DATASET_NAME -m $MAPS -i $SUBJECT_ID

In [ ]:
!ls $PATH_FREESURFER_DIR/$SUBJECT_ID/deepRetinotopy

In [ ]:
## Process single subject with an explicit output directory
!mkdir -p /home/jovyan/Desktop/deepRetinotopy_workshop/notebooks/data/output && \
deepRetinotopy -s $PATH_FREESURFER_DIR -t $PATH_HCP_TEMPLATE_SURFACE -d $DATASET_NAME -m $MAPS -i $SUBJECT_ID -o /home/jovyan/Desktop/deepRetinotopy_workshop/notebooks/data/output

In [ ]:
!ls /home/jovyan/Desktop/deepRetinotopy_workshop/notebooks/data/output/$SUBJECT_ID/

## Running deepRetinotopy for multiple subjects

If no subject is specified, all subjects within the FreeSurfer directory will be processed. However, for optimal speed, parallelising the processing for multiple subjects across multiple cores is preferred.

In [ ]:
## Determine how many cores you have available
os.environ['NUMBER_OF_CORES'] = str(os.cpu_count() - 1)

In [ ]:
%%bash

# SUBJECT_LIST=(sub-wlsubj001 sub-wlsubj004 sub-wlsubj006 sub-wlsubj007)
SUBJECT_LIST=("ADD MULTIPLE SUB IDs HERE AS ABOVE")
process_subject() {
    local subject=$1
    echo "Processing subject: $subject"
    deepRetinotopy -s $PATH_FREESURFER_DIR -t $PATH_HCP_TEMPLATE_SURFACE -d $DATASET_NAME -m $MAPS -i "$subject"
}

export -f process_subject

printf "%s\n" "${SUBJECT_LIST[@]}" | xargs -P "$NUMBER_OF_CORES" -I {} bash -c 'process_subject "{}"'

In [ ]:
!ls $PATH_FREESURFER_DIR/$SUBJECT_ID/deepRetinotopy

## Maps visualization

Finally, we can visualize the predicted maps in the individual's native space using nilearn.

In [ ]:
from nilearn import plotting
from ipywidgets import Dropdown, interact
import nibabel as nib
import numpy as np
from IPython.display import display

os.chdir(os.environ['PATH_FREESURFER_DIR'])

def plot_data(subject, retinotopic_map, surface = 'midthickness'):
    background = np.array(nib.load(f"./{subject}/surf/lh.graymid.H.gii").agg_data())[:]
    # binarizing curvature maps
    background[background < 0] = -1
    background[background > 1] = 1
    # setting a threshold to visualize the curvature maps in the background
    threshold = 1
    # loading the predicted retinotopic map and adding the threshold to non-zero values to visualize them on top of the curvature maps
    data = np.array(nib.load(f"./{subject}/deepRetinotopy/{subject}.predicted_{retinotopic_map}_model.lh.native.func.gii").agg_data())[:]
    data[data != 0] = data[data != 0] + threshold
    if retinotopic_map == 'eccentricity':
        max_value = 8 + threshold
    else: 
        max_value = 360 + threshold
    if surface == 'midthickness':
        mesh_file = f"./{subject}/surf/lh.midthickness.surf.gii"
    elif surface == 'sphere':
        mesh_file = f"./{subject}/surf/lh.sphere.reg.surf.gii"
        
    view = plotting.view_surf(
        surf_mesh= mesh_file,
        surf_map=np.reshape(data[:], (-1)),
        cmap="gist_rainbow_r", black_bg=False, symmetric_cmap=False, bg_map = background,
        threshold=threshold, vmax=max_value + threshold, vmin = threshold)
    return view


In [ ]:
subjects = [os.environ['SUBJECT_ID']]
retinotopic_maps = ['eccentricity', 'polarAngle']
surface = ['sphere', 'midthickness']

@interact(subject = subjects, retinotopic_map = retinotopic_maps, surface = surface)
def plot1(subject, retinotopic_map, surface):
    view = plot_data(subject, retinotopic_map, surface)
    display(view)